# Plan D — Anti-Overfitting Techniques

A collection of regularisation and data augmentation strategies to
reduce overfitting when training on a small dataset (2703 samples).

**Techniques covered:**

| # | Technique | Where | Impact |
|---|-----------|-------|--------|
| D1 | Feature-space data augmentation | DataLoader | +2-5x effective data |
| D2 | SpecAugment on DAC latents | DataLoader | Masks time/freq bands |
| D3 | Stronger weight decay | Config | Penalises large weights |
| D4 | Label smoothing (flow target noise) | Loss | Softer training targets |
| D5 | Gradient accumulation (larger eff. batch) | Training | Smoother gradients |
| D6 | Exponential Moving Average (EMA) | Model | Smoother weight trajectory |
| D7 | Learning rate tuning | Config | Slower convergence |

**How to use this notebook:**
- Each technique is implemented as an **independent, self-contained cell**
- You can mix and match techniques
- The final training cell applies ALL selected techniques together
- Checkpoints go to `checkpoints_regularised/` (separate from default)

**Prerequisites:** Same features as `train_colab.ipynb`. GPU runtime required.

---

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch, os
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f'GPU memory: {gpu_mem:.1f} GB')

PROJECT_DIR = "/content/speech-enhancement-project"

BRANCH       = "main"

if not os.path.exists(PROJECT_DIR):
    !git clone -b {BRANCH} https://github.com/VictorChen2002/Speech-Enhancement-Project.git {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull origin {BRANCH}

os.chdir(PROJECT_DIR)
!pip install -q -r requirements.txt

## 0.5 wandb Login

In [ ]:
import wandb
wandb.login()

## 1. Unpack Features (same as original)

In [ ]:
import os, glob
os.chdir(PROJECT_DIR)

ARCHIVES_DIR = "/content/drive/MyDrive/archives"
os.makedirs("data/features", exist_ok=True)

for name in ["features_clean_dac.tar.gz", "features_noisy_dac.tar.gz", "features_moss_last.tar.gz"]:
    feat = name.replace("features_", "").replace(".tar.gz", "")
    target = f"data/features/{feat}"
    if os.path.exists(target) and len(os.listdir(target)) > 0:
        print(f"✅ {feat} ({len(os.listdir(target))} files)")
        continue
    archive = os.path.join(ARCHIVES_DIR, name)
    if os.path.exists(archive):
        !tar xzf {archive} -C data/features/

# moss_multi shards
shard_pattern = os.path.join(ARCHIVES_DIR, "features_moss_multi_shard*.tar.gz")
shards = sorted(glob.glob(shard_pattern))
if shards:
    os.makedirs("data/features/moss_multi", exist_ok=True)
    for s in shards:
        !tar xzf {s} -C data/features/

print("\nFeature counts:")
for d in ['clean_dac', 'noisy_dac', 'moss_last', 'moss_multi']:
    p = f"data/features/{d}"
    n = len(os.listdir(p)) if os.path.exists(p) else 0
    print(f"  {d}: {n}")

## D1. Feature-Space Data Augmentation

Since our features are pre-computed DAC latents (not raw audio), we can't do
traditional audio augmentations. Instead, we augment in feature space:

1. **Gaussian noise injection**: Add small noise to DAC latents (X_0 and X_1)
2. **Random time shift**: Shift features by a few frames
3. **Random scaling**: Scale features by a random factor close to 1.0

These are applied **on-the-fly** during training, so we don't need extra storage.

In [ ]:
# D1: Feature-space augmentation wrapper
# This wraps the existing OfflineFeatureDataset with on-the-fly augmentations.

import torch
import random
from torch.utils.data import Dataset

class AugmentedFeatureDataset(Dataset):
    """Wraps OfflineFeatureDataset with feature-space augmentations."""

    def __init__(self, base_dataset, noise_std=0.01, scale_range=(0.95, 1.05),
                 time_shift_max=3, augment_prob=0.5):
        self.base = base_dataset
        self.noise_std = noise_std
        self.scale_range = scale_range
        self.time_shift_max = time_shift_max
        self.augment_prob = augment_prob

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        sample = self.base[idx]
        if random.random() > self.augment_prob:
            return sample  # No augmentation

        x0 = sample["noisy_dac"]   # (T, D) noisy input
        x1 = sample["clean_dac"]   # (T, D) clean target

        # 1. Gaussian noise injection
        if self.noise_std > 0:
            x0 = x0 + torch.randn_like(x0) * self.noise_std
            x1 = x1 + torch.randn_like(x1) * self.noise_std * 0.5  # less noise on target

        # 2. Random scaling
        lo, hi = self.scale_range
        scale = random.uniform(lo, hi)
        x0 = x0 * scale
        x1 = x1 * scale

        # 3. Random time shift (circular)
        if self.time_shift_max > 0:
            shift = random.randint(-self.time_shift_max, self.time_shift_max)
            if shift != 0:
                x0 = torch.roll(x0, shift, dims=0)
                x1 = torch.roll(x1, shift, dims=0)

        sample = dict(sample)  # shallow copy
        sample["noisy_dac"] = x0
        sample["clean_dac"] = x1
        return sample

print("D1: AugmentedFeatureDataset defined")
print("  noise_std=0.01, scale_range=(0.95, 1.05), time_shift_max=3, augment_prob=0.5")

## D2. SpecAugment on DAC Latents

Inspired by SpecAugment (for spectrograms), we mask random time steps and
feature dimensions in the DAC latent space. This forces the model to be
robust and not rely on any single time step or feature dimension.

In [ ]:
# D2: SpecAugment-style masking for DAC latents

import torch

class LatentSpecAugment:
    """Apply SpecAugment-style masking to DAC latent tensors."""

    def __init__(self, time_mask_max=10, freq_mask_max=64, num_time_masks=2,
                 num_freq_masks=2, mask_value=0.0):
        self.time_mask_max = time_mask_max    # max frames to mask
        self.freq_mask_max = freq_mask_max    # max feature dims to mask
        self.num_time_masks = num_time_masks
        self.num_freq_masks = num_freq_masks
        self.mask_value = mask_value

    def __call__(self, x):
        """x: (T, D) tensor"""
        T, D = x.shape
        x = x.clone()

        # Time masking
        for _ in range(self.num_time_masks):
            t = torch.randint(0, max(T - self.time_mask_max, 1), (1,)).item()
            t_len = torch.randint(1, self.time_mask_max + 1, (1,)).item()
            x[t:t+t_len, :] = self.mask_value

        # Feature masking
        for _ in range(self.num_freq_masks):
            f = torch.randint(0, max(D - self.freq_mask_max, 1), (1,)).item()
            f_len = torch.randint(1, self.freq_mask_max + 1, (1,)).item()
            x[:, f:f+f_len] = self.mask_value

        return x

print("D2: LatentSpecAugment defined")
print("  time_mask_max=10, freq_mask_max=64, num_time_masks=2, num_freq_masks=2")

## D3. Stronger Weight Decay & D7. Learning Rate Tuning

These are config-level changes that don't require code modifications.

- **Weight decay**: 1e-5 → 5e-4 (50x stronger L2 regularisation)
- **Learning rate**: 1e-4 → 5e-5 (slower convergence = less overfitting)
- **Dropout**: already at 0.1, bump to 0.15

In [ ]:
import yaml

with open('configs/default.yaml', 'r') as f:
    config = yaml.safe_load(f)

# ---- Regularisation overrides ----
config['training']['weight_decay'] = 5e-4       # D3: 50x stronger
config['training']['learning_rate'] = 5e-5       # D7: halved LR
config['model']['dropout'] = 0.15                # Higher dropout
config['training']['device'] = 'cuda'
config['training']['checkpoint_dir'] = 'checkpoints_regularised'

# GPU batch size
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3 if torch.cuda.is_available() else 0
if gpu_mem >= 35:
    config['training']['batch_size'] = 64
elif gpu_mem >= 14:
    config['training']['batch_size'] = 32
else:
    config['training']['batch_size'] = 16

# Display changes
print("Regularised config overrides:")
print(f"  weight_decay:   {config['training']['weight_decay']} (default: 1e-5)")
print(f"  learning_rate:  {config['training']['learning_rate']} (default: 1e-4)")
print(f"  dropout:        {config['model']['dropout']} (default: 0.1)")
print(f"  checkpoint_dir: {config['training']['checkpoint_dir']}")

with open('configs/colab_regularised.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("\nSaved configs/colab_regularised.yaml")

## D4. Flow Target Noise (Label Smoothing)

In flow matching, the target velocity is $V = X_1 - X_0$.
Adding small Gaussian noise to the target acts like label smoothing —
it prevents the model from memorising exact velocity vectors.

$$V_{\text{smooth}} = (X_1 - X_0) + \epsilon, \quad \epsilon \sim \mathcal{N}(0, \sigma^2)$$

We patch this into `RectifiedFlow.loss()` without modifying the original code.

In [ ]:
# D4: Smoothed flow loss

import torch
import torch.nn as nn

class SmoothedRectifiedFlow:
    """RectifiedFlow with label smoothing on the velocity target."""

    def __init__(self, target_noise_std=0.02):
        self.target_noise_std = target_noise_std

    def loss(self, model, x_0, x_1, cond=None, cond_layers=None):
        """
        MSE loss between predicted and ground-truth velocity field.
        Adds small noise to the target velocity for regularisation.
        """
        B = x_0.shape[0]
        t = torch.rand(B, device=x_0.device)
        t_expand = t[:, None, None]  # (B, 1, 1) for broadcasting

        # Straight-path interpolation
        x_t = t_expand * x_1 + (1 - t_expand) * x_0

        # Ground truth velocity
        v_target = x_1 - x_0

        # D4: Add noise to target (label smoothing)
        if self.target_noise_std > 0:
            v_target = v_target + torch.randn_like(v_target) * self.target_noise_std

        # Predict
        v_pred = model(x_t, t, cond=cond, cond_layers=cond_layers)
        return nn.functional.mse_loss(v_pred, v_target)

print("D4: SmoothedRectifiedFlow defined (target_noise_std=0.02)")

## D5. Gradient Accumulation

Simulates a larger batch size by accumulating gradients over multiple
forward passes before updating weights.

Effective batch size = `batch_size × accum_steps`

Larger effective batches produce smoother gradient estimates, which
helps with training stability and can reduce overfitting due to noisy
gradients on small datasets.

In [ ]:
# D5: Gradient accumulation is controlled by a single variable.
# The training loop below (Section 9) implements this.

GRADIENT_ACCUM_STEPS = 4  # Effective batch = 32 * 4 = 128

print(f"D5: Gradient accumulation steps = {GRADIENT_ACCUM_STEPS}")
print(f"     With batch_size=32, effective batch = {32 * GRADIENT_ACCUM_STEPS}")

## D6. Exponential Moving Average (EMA)

Maintains a shadow copy of model weights that's an exponential moving average
of the training weights. The EMA model typically generalises better because
it smooths out noisy weight updates.

$$\theta_{\text{EMA}} = \beta \cdot \theta_{\text{EMA}} + (1 - \beta) \cdot \theta$$

We use $\beta = 0.999$ (update every step, average over ~1000 steps).

In [ ]:
# D6: EMA wrapper for any nn.Module

import torch
import copy

class EMA:
    """Exponential Moving Average of model parameters."""

    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = copy.deepcopy(model)
        self.shadow.eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        """Update EMA weights."""
        for ema_p, model_p in zip(self.shadow.parameters(), model.parameters()):
            ema_p.data.mul_(self.decay).add_(model_p.data, alpha=1 - self.decay)

    def state_dict(self):
        return self.shadow.state_dict()

    def load_state_dict(self, state_dict):
        self.shadow.load_state_dict(state_dict)

print("D6: EMA class defined (decay=0.999)")
print("  Call ema.update(model) after each optimiser step")
print("  Use ema.shadow for inference/evaluation")

## 8. Combined Training with ALL Techniques

This cell runs a custom training loop that applies **all** the techniques above:

- **D1**: Feature-space augmentation (noise, scaling, time shift)
- **D2**: SpecAugment-style masking on noisy inputs
- **D3/D7**: Stronger weight decay + lower learning rate (via config)
- **D4**: Smoothed flow target (label smoothing)
- **D5**: Gradient accumulation (4 steps → 128 effective batch)
- **D6**: EMA model for better evaluation

**Configure which techniques to enable:**

In [ ]:
# ---- Enable/disable individual techniques ----
ENABLE_D1_AUGMENTATION    = True   # Feature-space augmentation
ENABLE_D2_SPECAUGMENT     = True   # SpecAugment masking
ENABLE_D4_LABEL_SMOOTH    = True   # Flow target noise
ENABLE_D5_GRAD_ACCUM      = True   # Gradient accumulation
ENABLE_D6_EMA             = True   # Exponential Moving Average
# D3/D7 are always active via the config file

# ---- Training parameters ----
CONDITION_TYPE = "none"   # Change to "last_layer" or "multi_layer"
DRIVE_CKPT_DIR = "/content/drive/MyDrive/speech_enhancement_checkpoints_regularised"
WANDB_PROJECT  = "speech-enhancement"

print("Enabled techniques:")
print(f"  D1 Augmentation:    {ENABLE_D1_AUGMENTATION}")
print(f"  D2 SpecAugment:     {ENABLE_D2_SPECAUGMENT}")
print(f"  D3 Weight decay:    Always (via config)")
print(f"  D4 Label smoothing: {ENABLE_D4_LABEL_SMOOTH}")
print(f"  D5 Grad accum:      {ENABLE_D5_GRAD_ACCUM} (steps={GRADIENT_ACCUM_STEPS})")
print(f"  D6 EMA:             {ENABLE_D6_EMA}")
print(f"  D7 Low LR:          Always (via config)")
print(f"  Condition type:     {CONDITION_TYPE}")

In [ ]:
import os, sys, math, json, shutil, random
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import DataLoader
from tqdm import tqdm
import yaml

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

from src.models.dit import DiffusionTransformer
from src.models.flow_matching import RectifiedFlow

# ---- Load config ----
with open('configs/colab_regularised.yaml') as f:
    config = yaml.safe_load(f)

cfg_data = config['data']
cfg_model = config['model']
cfg_train = config['training']

condition_type = CONDITION_TYPE
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Seed
seed = cfg_train['seed']
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# ---- Load dataset ----
# Import the dataset class from train.py
from train import OfflineFeatureDataset

features_dir = cfg_data['features_dir']
split_file = cfg_data.get('split_file', 'data/split.json')

with open(split_file) as f:
    split = json.load(f)

train_dataset = OfflineFeatureDataset(
    features_dir, condition_type=condition_type,
    max_seq_len=cfg_data['max_seq_len'], stems=split['train'],
)
valid_dataset = OfflineFeatureDataset(
    features_dir, condition_type=condition_type,
    max_seq_len=cfg_data['max_seq_len'], stems=split['valid'],
)

# D1: Wrap with augmentation
if ENABLE_D1_AUGMENTATION:
    train_dataset = AugmentedFeatureDataset(
        train_dataset, noise_std=0.01, scale_range=(0.95, 1.05),
        time_shift_max=3, augment_prob=0.5,
    )
    print(f"D1: Augmented dataset wrapping {len(train_dataset)} samples")

# D2: SpecAugment transform
spec_augment = LatentSpecAugment(
    time_mask_max=10, freq_mask_max=64,
    num_time_masks=2, num_freq_masks=2,
) if ENABLE_D2_SPECAUGMENT else None

batch_size = cfg_train['batch_size']
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset)} samples, Valid: {len(valid_dataset)} samples")
print(f"Batch size: {batch_size}")

# ---- Build model ----
num_moss_layers = cfg_model.get('num_moss_layers', 32)
moss_embed_dim = cfg_model.get('moss_embed_dim', 'auto')

if condition_type == "multi_layer":
    moss_multi_dir = Path(features_dir) / "moss_multi"
    sample_files = sorted(moss_multi_dir.glob("*.pt"))
    if sample_files:
        sample = torch.load(str(sample_files[0]), map_location="cpu", weights_only=False)
        num_moss_layers = len(sample)
        if moss_embed_dim == "auto":
            moss_embed_dim = sample[0].shape[-1]
elif condition_type == "last_layer":
    moss_last_dir = Path(features_dir) / "moss_last"
    sample_files = sorted(moss_last_dir.glob("*.pt"))
    if sample_files and moss_embed_dim == "auto":
        s = torch.load(str(sample_files[0]), map_location="cpu", weights_only=False)
        moss_embed_dim = s.shape[-1]

if moss_embed_dim == "auto":
    moss_embed_dim = 768

model = DiffusionTransformer(
    dac_latent_dim=cfg_model['dac_latent_dim'],
    moss_embed_dim=moss_embed_dim,
    hidden_dim=cfg_model['hidden_dim'],
    num_heads=cfg_model['num_heads'],
    num_layers=cfg_model['num_layers'],
    dropout=cfg_model['dropout'],
    condition_type=condition_type,
    num_moss_layers=num_moss_layers,
).to(device)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {num_params:,} trainable parameters")

# D6: EMA
ema = EMA(model, decay=0.999) if ENABLE_D6_EMA else None

# D4: Flow loss
if ENABLE_D4_LABEL_SMOOTH:
    flow = SmoothedRectifiedFlow(target_noise_std=0.02)
    print("D4: Using smoothed flow loss (target_noise_std=0.02)")
else:
    flow = RectifiedFlow()

# ---- Optimiser & Scheduler ----
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg_train['learning_rate'],
    weight_decay=cfg_train['weight_decay'],
)

warmup_steps = cfg_train.get('warmup_steps', 0)
num_epochs = cfg_train.get('num_epochs', 100)
steps_per_epoch = math.ceil(len(train_dataset) / batch_size)
total_steps = num_epochs * steps_per_epoch
patience = cfg_train.get('patience', 20)

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# D5: Gradient accumulation
accum_steps = GRADIENT_ACCUM_STEPS if ENABLE_D5_GRAD_ACCUM else 1

# ---- Checkpoint dir ----
ckpt_dir = Path(cfg_train['checkpoint_dir']) / condition_type
ckpt_dir.mkdir(parents=True, exist_ok=True)

if DRIVE_CKPT_DIR:
    drive_ct_dir = Path(DRIVE_CKPT_DIR) / condition_type
    drive_ct_dir.mkdir(parents=True, exist_ok=True)

# ---- wandb ----
import wandb
techniques = []
if ENABLE_D1_AUGMENTATION: techniques.append("D1")
if ENABLE_D2_SPECAUGMENT: techniques.append("D2")
techniques.append("D3")  # always
if ENABLE_D4_LABEL_SMOOTH: techniques.append("D4")
if ENABLE_D5_GRAD_ACCUM: techniques.append("D5")
if ENABLE_D6_EMA: techniques.append("D6")
techniques.append("D7")  # always

run_name = f"reg_{condition_type}_{'_'.join(techniques)}"
wandb_run = wandb.init(
    project=WANDB_PROJECT,
    name=run_name,
    config={
        **config,
        "condition_type": condition_type,
        "techniques": techniques,
        "accum_steps": accum_steps,
        "ema_decay": 0.999 if ENABLE_D6_EMA else None,
        "target_noise_std": 0.02 if ENABLE_D4_LABEL_SMOOTH else 0,
    },
)

print(f"\n[wandb] {wandb_run.url}")
print(f"[Schedule] {num_epochs} epochs, {steps_per_epoch} steps/epoch, total={total_steps}")
if ENABLE_D5_GRAD_ACCUM:
    print(f"[D5] Gradient accumulation: {accum_steps} steps (eff. batch={batch_size * accum_steps})")

# ---- Training loop ----
model.train()
global_step = 0
best_val_loss = float("inf")
no_improve_count = 0

for epoch in range(1, num_epochs + 1):
    epoch_loss = 0.0
    epoch_batches = 0
    optimizer.zero_grad()

    for batch_idx, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs}", leave=False)):
        x0 = batch["noisy_dac"].to(device)
        x1 = batch["clean_dac"].to(device)

        # D2: SpecAugment on noisy input
        if spec_augment is not None:
            for i in range(x0.shape[0]):
                x0[i] = spec_augment(x0[i])

        cond = batch.get("moss_last", None)
        cond_layers = batch.get("moss_multi", None)
        if cond is not None:
            cond = cond.to(device)
        if cond_layers is not None:
            cond_layers = [c.to(device) for c in cond_layers]

        loss = flow.loss(model, x0, x1, cond=cond, cond_layers=cond_layers)
        loss = loss / accum_steps  # D5: scale loss
        loss.backward()

        epoch_loss += loss.item() * accum_steps
        epoch_batches += 1

        # D5: Only step every accum_steps
        if (batch_idx + 1) % accum_steps == 0 or (batch_idx + 1) == len(train_loader):
            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(), cfg_train.get('gradient_clip', 1.0)
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            # D6: EMA update
            if ema is not None:
                ema.update(model)

            # Log
            if global_step % cfg_train.get('log_every', 50) == 0:
                wandb.log({
                    "train/loss": loss.item() * accum_steps,
                    "train/lr": scheduler.get_last_lr()[0],
                    "train/grad_norm": grad_norm.item(),
                }, step=global_step)

    avg_train_loss = epoch_loss / max(epoch_batches, 1)

    # ---- Validation ----
    model.eval()
    val_loss_sum = 0.0
    val_batches = 0
    flow_eval = RectifiedFlow()  # Use non-smoothed loss for validation

    with torch.no_grad():
        for batch in valid_loader:
            x0 = batch["noisy_dac"].to(device)
            x1 = batch["clean_dac"].to(device)
            cond = batch.get("moss_last", None)
            cond_layers = batch.get("moss_multi", None)
            if cond is not None:
                cond = cond.to(device)
            if cond_layers is not None:
                cond_layers = [c.to(device) for c in cond_layers]
            loss = flow_eval.loss(model, x0, x1, cond=cond, cond_layers=cond_layers)
            val_loss_sum += loss.item()
            val_batches += 1

    val_loss = val_loss_sum / max(val_batches, 1)
    model.train()

    lr = scheduler.get_last_lr()[0]
    print(f"[Epoch {epoch:>3d}/{num_epochs}]  train={avg_train_loss:.6f}  val={val_loss:.6f}  lr={lr:.2e}")

    wandb.log({
        "epoch/train_loss": avg_train_loss,
        "epoch/val_loss": val_loss,
        "epoch/lr": lr,
        "epoch": epoch,
    }, step=global_step)

    # ---- Save checkpoint ----
    ckpt_path = ckpt_dir / f"step_{global_step}.pt"
    payload = {
        "step": global_step,
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "config": config,
        "condition_type": condition_type,
        "best_val_loss": best_val_loss,
        "no_improve_count": no_improve_count,
    }
    if ema is not None:
        payload["ema_state_dict"] = ema.state_dict()
    torch.save(payload, str(ckpt_path))

    if DRIVE_CKPT_DIR:
        dst = Path(DRIVE_CKPT_DIR) / condition_type / f"step_{global_step}.pt"
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(str(ckpt_path), str(dst))

    # ---- Track best ----
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve_count = 0
        best_dst = ckpt_dir / "best.pt"
        shutil.copy2(str(ckpt_path), str(best_dst))
        if DRIVE_CKPT_DIR:
            shutil.copy2(str(ckpt_path), str(Path(DRIVE_CKPT_DIR) / condition_type / "best.pt"))
        # Also save EMA as best
        if ema is not None:
            ema_path = ckpt_dir / "best_ema.pt"
            torch.save(ema.state_dict(), str(ema_path))
            if DRIVE_CKPT_DIR:
                shutil.copy2(str(ema_path), str(Path(DRIVE_CKPT_DIR) / condition_type / "best_ema.pt"))
        print(f"  ★ New best val_loss={val_loss:.6f} (epoch {epoch})")
    else:
        no_improve_count += 1
        print(f"  val_loss did not improve ({no_improve_count}/{patience})")

    if patience > 0 and no_improve_count >= patience:
        print(f"\n⏹  Early stopping at epoch {epoch}")
        break

wandb.finish()
print(f"\nTraining complete. Final step: {global_step}")

## 9. Evaluate Regularised Model

Evaluate the regularised model and compare with the default model.

In [ ]:
import os
os.chdir(PROJECT_DIR)

# Update config to point to regularised checkpoints
import yaml
with open('configs/colab_regularised.yaml') as f:
    config_reg = yaml.safe_load(f)

# Compare the regularised model
cmd = f"python evaluate.py --config configs/colab_regularised.yaml --compare"
cmd += f" --drive_ckpt_dir {DRIVE_CKPT_DIR}"

print(f"Command:\n  {cmd}\n")
!{cmd}

## 10. Disconnect Runtime

In [ ]:
from google.colab import runtime
runtime.unassign()